In [ ]:
import glob
import re
import pandas as pd

def summarize_pbs_outputs(path, file_pattern="*.o*"):
    """
    Summarize PBS output files in a directory.

    Parameters
    ----------
    path : str
        Directory containing PBS output files.
    file_pattern : str
        Glob pattern to match PBS output files.

    Returns
    -------
    df : pd.DataFrame
        DataFrame containing parsed PBS outputs.
    averages : pd.Series
        Average values of numeric columns.
    """

    all_files = sorted(glob.glob(f"{path}/{file_pattern}"))
    files = [f for f in all_files if "postscript" not in f.lower()]
    data_list = []

    # Regex patterns
    patterns = {
        "JobId": r"Job Id:\s*(\S+)",
        "ExitStatus": r"Exit Status:\s*(\d+)",
        "ServiceUnits": r"Service Units:\s*([\d\.]+)",
        "NCPUsUsed": r"NCPUs Used:\s*(\d+)",
        "CPUTime": r"CPU Time Used:\s*(\d+):(\d+):(\d+)",
        "MemoryUsedGB": r"Memory Used:\s*([\d\.]+)GB",
        "WalltimeUsed": r"Walltime Used:\s*(\d+):(\d+):(\d+)"
    }

    for f in files:
        with open(f, "r") as file:
            text = file.read()
            row = {}
            for key, pat in patterns.items():
                m = re.search(pat, text)
                if m:
                    if key in ["CPUTime", "WalltimeUsed"]:
                        h, m_, s = map(int, m.groups())
                        row[key] = h*3600 + m_*60 + s  # seconds
                    elif key in ["ExitStatus", "NCPUsUsed", "MemoryUsedGB", "ServiceUnits"]:
                        row[key] = float(m.group(1)) if '.' in m.group(1) else int(m.group(1))
                    else:  # JobId or other strings
                        row[key] = m.group(1)
                else:
                    row[key] = None
            row["Filename"] = f.split("/")[-1]
            data_list.append(row)

    # Build DataFrame
    df = pd.DataFrame(data_list)
    
    # Remove failed jobs (ExitStatus == 1)
    # df = df[df["ExitStatus"] != 1]
    
    # Compute averages (median, as you're using)
    averages = df.select_dtypes(include="number").median()

    return df, averages

# Read in data

In [ ]:
# List of experiment directories
experiment_paths = [
    "/g/data/ps29/nd0349/access-om3/configurations/mcw/IC4M8-MCW-100km_jra_iaf_2010",
    "/g/data/ps29/nd0349/access-om3/configurations/mcw/IC4M8-MCW-100km_jra_iaf_2010-constant",
    "/g/data/ps29/nd0349/access-om3/configurations/mcw/IC4M8-MCW-100km_jra_iaf_2010-nofrac",
    "/g/data/ps29/nd0349/access-om3/configurations/mcw/MCW-100km_jra_iaf-2010-icdr",
    "/g/data/ps29/nd0349/access-om3/configurations/mcw/MCW-100km_jra_iaf-2010-icdr-test",
    "/g/data/ps29/nd0349/access-om3/configurations/MC_100km_era_iaf",
    # "/g/data/ps29/nd0349/access-om3/configurations/mcw/MCW-100km_jra_iaf-2010-icdr-weld",
    "/g/data/ps29/nd0349/access-om3/configurations/mcw/MCW_100km_era_iaf",
    "/g/data/ps29/nd0349/access-om3/configurations/mcw/MCW_100km_era_iaf_random",
    # "/g/data/ps29/nd0349/access-om3/configurations/ww3/WW3-standalone-ERA5-dice",
    "/g/data/ps29/nd0349/access-om3/configurations/ww3/WW3-standalone-ERA5-dice-2013",
    # "/g/data/ps29/nd0349/access-om3/configurations/ww3/WW3-standalone-ERA5-dice",
    # "/scratch/ps29/nd0349/access-om3/archive/MCW-100km_jra_iaf-2010-icdr/pbs_logs",
    # "/scratch/ps29/nd0349/access-om3/archive/MCW_100km_era_iaf_KPP/pbs_logs",
    # "/scratch/ps29/nd0349/access-om3/archive/MCW-100km_jra_iaf-2010-icdr-weld/pbs_logs",
    # "/g/data/ps29/nd0349/access-om3/configurations/mcw/MCW-100km_jra_iaf-2010-icdr-weld",
    
]
target_period = "yearly"   # choose "monthly" or "yearly"

def get_exp_name(path):
    parts = path.rstrip("/").split("/")
    if parts[-1] == "pbs_logs":
        return parts[-2]
    return parts[-1]

def get_source_period(exp_name):
    if exp_name == "MC_100km_era_iaf":
        return "yearly"
    return "monthly"

def period_factor(source_period, target_period):
    if source_period == target_period:
        return 1
    if source_period == "monthly" and target_period == "yearly":
        return 12
    if source_period == "yearly" and target_period == "monthly":
        return 1 / 12
    raise ValueError(f"Unknown conversion: {source_period} -> {target_period}")

# Dictionary to store results
all_averages = {}

for path in experiment_paths:
    df, averages = summarize_pbs_outputs(path)
    averages = averages.copy()

    exp_name = get_exp_name(path)

    source_period = get_source_period(exp_name)
    factor = period_factor(source_period, target_period)

    # Convert CPUTime and WalltimeUsed to hours
    if "CPUTime" in averages:
        averages["CPUTime"] = averages["CPUTime"] / 3600
    if "WalltimeUsed" in averages:
        averages["WalltimeUsed"] = averages["WalltimeUsed"] / 3600
    if "ServiceUnits" in averages:
        averages["ServiceUnits"] = averages["ServiceUnits"] / 1000

    # Scale accumulated metrics only
    for metric in ["CPUTime", "WalltimeUsed", "ServiceUnits"]:
        if metric in averages:
            averages[metric] = averages[metric] * factor

    all_averages[exp_name] = averages
    

# Display results
for exp, avg in all_averages.items():
    print(f"\nExperiment: {exp}")
    print(avg)

# Plot median results

In [ ]:
import matplotlib.pyplot as plt
import os
import seaborn as sns

sns.set_style("ticks")

# Example: all_averages.keys() are your experiment names
exp_names = list(all_averages.keys())

# Detect common prefix
common_prefix = os.path.commonprefix(exp_names)

# Terms to remove
remove_terms = ["IC4M8", "MCW", "jra", "100km", "2010", "2013", "iaf", "pbs", "logs"]

# Remove these terms and clean up labels
short_labels = []
for name in exp_names:
    clean_name = name
    for term in remove_terms:
        clean_name = clean_name.replace(term, "")
    # Remove extra hyphens/underscores and capitalize
    parts = [p.capitalize() for p in clean_name.replace("_", "-").split("-") if p]
    short_labels.append("".join(parts) if parts else "Random")

# Metrics to plot
metrics = ["MemoryUsedGB", "CPUTime", "WalltimeUsed", "ServiceUnits"]

# Prepare figure
fig, axes = plt.subplots(1, len(metrics), figsize=(4*len(metrics), 3), constrained_layout=True)

# Plot each metric
for ax, metric in zip(axes, metrics):
    values = [all_averages[exp][metric] for exp in exp_names]
    ax.bar(short_labels, values)
    ax.set_ylabel(metric)
    # ax.set_title(f"Average {metric}")
    ax.set_ylim(0, 1.2*max(values))
    # Add numeric labels on top of bars
    for i, val in enumerate(values):
        ax.text(i, val*1.00, f"{val:.2f}", ha='center', va='bottom', fontsize=9)
        
    ax.set_xticklabels(short_labels, rotation=30, ha='right')
    ax.grid(True, axis='y')  # only horizontal lines
plt.show()